Script for inference with Llama-3.2 using HuggingFace API

[Login to HuggingFace Hub to use model with requested access](https://huggingface.co/docs/huggingface_hub/en/quick-start#authentication)

In [30]:
from huggingface_hub import login
login()

Generate data with LlaMa

In [31]:
from pathlib import Path
from transformers import pipeline, AutoTokenizer, AutoModelForCausalLM
from tqdm.auto import tqdm

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent

model_name = "meta-llama/Llama-3.2-1B-Instruct"

# Load tokenizer with left padding
tokenizer = AutoTokenizer.from_pretrained(model_name, padding_side="left")
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token 
model = AutoModelForCausalLM.from_pretrained(model_name, device_map="auto")
    
generator = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    max_new_tokens=500,
    device_map="auto"
)

Device set to use cuda:0


In [32]:
def counselor_response(questions): 
    
    messages = [[
        {"role": "system", "content": "Provide a concise and clear answer to the user's question. Don't leave a blank answer"},
        {"role": "user", "content": question}
    ] for question in questions]
    output = generator(messages)
    
    return output

In [33]:
import torch
torch.cuda.empty_cache()
torch.cuda.ipc_collect()

Use batch inference to save time + GPU capacity. Suggest experimenting with batch size to get desirable results.

In [ ]:
from datasets import load_dataset
import time

questions = load_dataset("rileyhitthefan/age-based-health-qa")["train"]
outputs =  []

batch_size = 100
count = 1

for i in range(0, len(questions), batch_size):
    print("processing batch " + str(count))
    start_time = time.time()
    output = counselor_response(questions['prompt'][i:i+batch_size])
    print((time.time() - start_time)//60)
    count += 1
    outputs.append(output)

processing batch 1
10.0
processing batch 2
11.0
processing batch 3
10.0
processing batch 4
11.0
processing batch 5
11.0
processing batch 6
10.0
processing batch 7
10.0


In [35]:
len(outputs) # 10
# batch - item in batch - content in item
outputs[0][0] # {'generated_text':...}
outputs[0][1][0]['generated_text'] # {system, user, assistant}

[{'role': 'system',
  'content': "Provide a concise and clear answer to the user's question. Don't leave a blank answer"},
 {'role': 'user',
  'content': "I'm 17 and I've been hanging out with some friends who work on farms. I heard they might have been exposed to something called brucellosis. Is it possible to catch it from them or from animals?"},
 {'role': 'assistant',
  'content': "Brucellosis is a bacterial infection that can be spread through contact with infected animals or their products, such as milk, meat, or bone marrow. It's also possible to contract the disease from infected animals that come into contact with humans, such as through handling or petting.\n\nIn the case of farm workers, they may be at higher risk of exposure to brucellosis due to their close contact with animals and their handling of potentially infected products. However, it's essential to note that the risk of infection is relatively low if proper hygiene and safety precautions are followed.\n\nSome ways 

In [36]:
import pandas as pd

questions = []
responses = []

for batch in outputs:
    for item in batch:
        qns = item[0]['generated_text'][1].get('content')
        questions.append(qns)
        ans = item[0]['generated_text'][2].get('content')
        responses.append(ans)

response_df = pd.DataFrame(zip(questions, responses), columns=['question','response_llama'])
response_df.head()

,question,response
0,Is brucellosis contagious?,"Yes, brucellosis is a contagious bacterial dis..."
1,I'm 17 and I've been hanging out with some fri...,Brucellosis is a bacterial infection that can ...
2,I'm 32 and I work at a veterinary clinic. I've...,"Brucellosis is a zoonotic disease, meaning it ..."
3,I'm 52 and I've been helping with a farm-relat...,"Brucellosis is a zoonotic disease, which means..."
4,I'm 74 and I used to work on a farm in my youn...,I can provide you with accurate and reliable i...


In [ ]:
import os

os.makedirs(ROOT / "data/responses", exist_ok=True)
response_df.to_csv(ROOT / "data/responses/response_llama.csv", index=False)